# Reproducing the results of the manuscript

This notebook recomputes the numbers of the manuscript from the result files in `results/`, which
are the outputs of the original runs (see `results/README.md` and `docs/PROVENANCE.md`):

| Manuscript | Recomputed from |
|---|---|
| Table 1, Supplementary Table S1, Fig. 4 | per-profile test predictions of the five MethylLlama and the five MethylGPT folds (2,149 profiles) |
| Fig. 2 | cross-view statistics of all 16,912 held-out pretraining profiles; reconstruction errors of 5,000 profiles |
| Fig. 3, Supplementary Table S2 | frozen CLS and mean-pooled representations of all 10,988 downstream profiles |

No GPU, model or data download is needed; it runs in about five minutes on a laptop CPU. Every value is compared with
the value printed in the manuscript. The figures are drawn from the same files by the scripts in
`scripts/figures/`. Producing the result files themselves from the public data is described in the
README ("Reproducing the manuscript").

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import balanced_accuracy_score, r2_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler

ROOT = Path.cwd().parent                     # run the notebook from tutorials/
RES = ROOT / "results"
AGE = RES / "age_benchmark"
PRE = RES / "pretrained_cls"

checks = []

def check(name, value, manuscript, decimals):
    # A value matches when it rounds to the number printed in the manuscript.
    ok = round(float(value), decimals) == round(float(manuscript), decimals)
    checks.append({"quantity": name, "recomputed": round(float(value), decimals + 2),
                   "manuscript": manuscript, "match": "yes" if ok else "NO"})

def report(first):
    print(pd.DataFrame(checks[first:]).to_string(index=False))

## 1. Age prediction (Table 1, Supplementary Table S1, Fig. 4)

Each ensemble averages the five fold predictions of a test profile. The MethylGPT folds were
fine-tuned from the authors' released checkpoint with their pipeline on the same profiles and
folds. ElasticNetCV profile-level predictions were not retained, so its row is read from the
run summary.

In [2]:
start = len(checks)
def read_folds(model):
    return [pd.read_csv(AGE / f"{model}/fold_{k}_predictions.csv", dtype={"sample_id": str}).set_index("sample_id")
            for k in range(5)]

folds, gpt_folds = read_folds("methyllama"), read_folds("methylgpt")
paired = pd.read_csv(AGE / "paired_per_subject_predictions.csv", dtype={"sample_id": str}).set_index("sample_id")
assert len(paired) == 2149 and all(len(f) == 2149 for f in folds + gpt_folds)

# the paired file holds the two five-fold ensembles
for fs, column in [(folds, "predicted_age_llama"), (gpt_folds, "predicted_age_gpt")]:
    assert np.allclose(sum(f.loc[paired.index, "predicted_age"] for f in fs) / 5, paired[column], atol=1e-9)
true = paired["true_age"].to_numpy()

def metrics(t, p):
    return np.median(np.abs(p - t)), np.mean(np.abs(p - t)), r2_score(t, p)

def fold_metrics(fs):
    return np.array([metrics(f.loc[paired.index, "true_age"].to_numpy(), f.loc[paired.index, "predicted_age"].to_numpy())
                     for f in fs])

per_fold, gpt_per_fold = fold_metrics(folds), fold_metrics(gpt_folds)
llama = metrics(true, paired["predicted_age_llama"].to_numpy())
gpt = metrics(true, paired["predicted_age_gpt"].to_numpy())
enet = json.loads((AGE / "elasticnet_results.json").read_text())["test"]

for k, paper in enumerate([(3.126, 4.439, 0.9321), (3.213, 4.458, 0.9337), (3.209, 4.570, 0.9284),
                           (3.152, 4.541, 0.9298), (3.144, 4.455, 0.9338)]):
    for name, v, p, d in zip(("MedAE", "MAE", "R2"), per_fold[k], paper, (3, 3, 4)):
        check(f"MethylLlama fold {k} {name}", v, p, d)
for name, j, p, d in [("MedAE", 0, (3.169, 0.040), 3), ("MAE", 1, (4.493, 0.059), 3), ("R2", 2, (0.9316, 0.0024), 4)]:
    check(f"MethylLlama fold mean {name}", per_fold[:, j].mean(), p[0], d)
    check(f"MethylLlama fold SD {name}", per_fold[:, j].std(ddof=1), p[1], d)
for k, paper in enumerate([(4.195, 5.855, 0.8846), (4.129, 5.891, 0.8825), (4.052, 5.705, 0.8907),
                           (4.162, 5.748, 0.8897), (3.971, 5.689, 0.8922)]):
    for name, v, p, d in zip(("MedAE", "MAE", "R2"), gpt_per_fold[k], paper, (3, 3, 4)):
        check(f"MethylGPT fold {k} {name}", v, p, d)
for name, j, p, d in [("MedAE", 0, (4.102, 0.090), 3), ("MAE", 1, (5.777, 0.091), 3), ("R2", 2, (0.8880, 0.0042), 4)]:
    check(f"MethylGPT fold mean {name}", gpt_per_fold[:, j].mean(), p[0], d)
    check(f"MethylGPT fold SD {name}", gpt_per_fold[:, j].std(ddof=1), p[1], d)
for model, m, paper in [("MethylLlama ensemble", llama, (3.112, 4.333, 0.9365)),
                        ("MethylGPT ensemble", gpt, (3.701, 5.105, 0.9110)),
                        ("ElasticNetCV", (enet["medae"], enet["mae"], enet["r2"]), (3.437, 4.731, 0.9322))]:
    for name, v, p, d in zip(("MedAE", "MAE", "R2"), m, paper, (3, 3, 4)):
        check(f"{model} {name}", v, p, d)
check("ElasticNetCV non-zero coefficients", json.loads((AGE / "elasticnet_results.json").read_text())["n_nonzero_coef"], 1036, 0)
for name, v, p, d in zip(("MedAE", "MAE", "R2"),
                         (enet["medae"] - llama[0], enet["mae"] - llama[1], llama[2] - enet["r2"]),
                         (0.325, 0.398, 0.0043), (3, 3, 4)):
    check(f"improvement over ElasticNetCV {name}", v, p, d)
report(start)

                           quantity  recomputed  manuscript match
           MethylLlama fold 0 MedAE    3.126300      3.1260   yes
             MethylLlama fold 0 MAE    4.439260      4.4390   yes
              MethylLlama fold 0 R2    0.932132      0.9321   yes
           MethylLlama fold 1 MedAE    3.212550      3.2130   yes
             MethylLlama fold 1 MAE    4.457620      4.4580   yes
              MethylLlama fold 1 R2    0.933656      0.9337   yes
           MethylLlama fold 2 MedAE    3.209440      3.2090   yes
             MethylLlama fold 2 MAE    4.570260      4.5700   yes
              MethylLlama fold 2 R2    0.928390      0.9284   yes
           MethylLlama fold 3 MedAE    3.151620      3.1520   yes
             MethylLlama fold 3 MAE    4.540970      4.5410   yes
              MethylLlama fold 3 R2    0.929793      0.9298   yes
           MethylLlama fold 4 MedAE    3.143780      3.1440   yes
             MethylLlama fold 4 MAE    4.454530      4.4550   yes
          

### Paired bootstrap against MethylGPT

10,000 resamples of the 2,149 test profiles (seed 0), the same resample applied to both models.
Positive values favour MethylLlama: error differences are MethylGPT minus MethylLlama, the $R^2$
difference is MethylLlama minus MethylGPT. The nested age restrictions (Fig. 4e) repeat the
procedure within each subset.

In [3]:
start = len(checks)

def paired_bootstrap(t, a, b, n_boot=10000, seed=0):
    # a = MethylLlama, b = MethylGPT; returns observed differences and 95% percentile intervals
    rng = np.random.default_rng(seed)
    diff = lambda i: (lambda ma, mb: (mb[0] - ma[0], mb[1] - ma[1], ma[2] - mb[2]))(metrics(t[i], a[i]), metrics(t[i], b[i]))
    obs = diff(np.arange(len(t)))
    boot = np.array([diff(rng.integers(0, len(t), size=len(t))) for _ in range(n_boot)])
    return obs, np.percentile(boot, [2.5, 97.5], axis=0).T

a, b = paired["predicted_age_llama"].to_numpy(), paired["predicted_age_gpt"].to_numpy()
obs, ci = paired_bootstrap(true, a, b)
for j, (name, d, paper) in enumerate([("MedAE", 3, (0.589, 0.338, 0.815)), ("MAE", 3, (0.773, 0.583, 0.968)),
                                      ("R2", 4, (0.0254, 0.0183, 0.0329))]):
    check(f"paired improvement {name}", obs[j], paper[0], d)
    check(f"  95% CI low", ci[j, 0], paper[1], d)
    check(f"  95% CI high", ci[j, 1], paper[2], d)

for min_age, paper in [(18, (1776, 0.680, 0.417, 0.948)), (40, (1449, 0.636, 0.265, 0.920))]:
    s = true >= min_age
    obs_s, ci_s = paired_bootstrap(true[s], a[s], b[s])
    check(f"age >= {min_age}: n", s.sum(), paper[0], 0)
    check(f"age >= {min_age}: paired MedAE improvement", obs_s[0], paper[1], 3)
    check(f"  95% CI low", ci_s[0, 0], paper[2], 3)
    check(f"  95% CI high", ci_s[0, 1], paper[3], 3)
report(start)

                           quantity  recomputed  manuscript match
           paired improvement MedAE    0.588520      0.5890   yes
                         95% CI low    0.337980      0.3380   yes
                        95% CI high    0.814890      0.8150   yes
             paired improvement MAE    0.772520      0.7730   yes
                         95% CI low    0.583100      0.5830   yes
                        95% CI high    0.968250      0.9680   yes
              paired improvement R2    0.025445      0.0254   yes
                         95% CI low    0.018342      0.0183   yes
                        95% CI high    0.032950      0.0329   yes
                       age >= 18: n 1776.000000   1776.0000   yes
age >= 18: paired MedAE improvement    0.679510      0.6800   yes
                         95% CI low    0.417370      0.4170   yes
                        95% CI high    0.948400      0.9480   yes
                       age >= 40: n 1449.000000   1449.0000   yes
age >= 40:

### Fold 0: pretrained versus random initialisation (Fig. 4c)

Validation trajectories of the two fold-0 runs, read from their training logs.

In [4]:
start = len(checks)
pre = pd.read_csv(AGE / "fold0_train_history.csv").set_index("epoch")
rnd = pd.read_csv(AGE / "random_init_fold0_history.csv").set_index("epoch")
check("epoch 148 validation MedAE, pretrained", pre.loc[148, "val/medae"], 2.875, 3)
check("epoch 148 validation MedAE, random init", rnd.loc[148, "val/medae"], 3.969, 3)
check("epoch 148 validation R2, pretrained", pre.loc[148, "val/r2"], 0.931, 3)
check("epoch 148 validation R2, random init", rnd.loc[148, "val/r2"], 0.854, 3)
check("best validation MedAE, pretrained", pre["val/medae"].min(), 2.625, 3)
check("  at epoch", pre["val/medae"].idxmin(), 92, 0)
check("best validation MedAE, random init", rnd["val/medae"].min(), 3.804, 3)
check("  at epoch", rnd["val/medae"].idxmin(), 187, 0)
report(start)

                               quantity  recomputed  manuscript match
 epoch 148 validation MedAE, pretrained     2.87500       2.875   yes
epoch 148 validation MedAE, random init     3.96875       3.969   yes
    epoch 148 validation R2, pretrained     0.93119       0.931   yes
   epoch 148 validation R2, random init     0.85407       0.854   yes
      best validation MedAE, pretrained     2.62500       2.625   yes
                               at epoch    92.00000      92.000   yes
     best validation MedAE, random init     3.80428       3.804   yes
                               at epoch   187.00000     187.000   yes


## 2. Cross-view consistency and reconstruction (Fig. 2)

For every draw, the per-profile arrays hold the cosine similarity of each profile's two views and
the rank of its partner among all candidates (rank 0 = nearest neighbour). Top-1 retrieval is the
fraction of rank 0. The unmatched similarity is the mean over all $16{,}912 \times 16{,}911$
non-partner pairs, which were not stored individually; it is read from the run records.

In [5]:
start = len(checks)
VC = PRE / "view_consistency_heldout_full/per_run"
runs = {}
for f in sorted(VC.glob("*_seed*.json")):
    j, z = json.loads(f.read_text()), np.load(f.with_suffix(".npz"))
    runs.setdefault(j["condition"], []).append({
        "matched": z["all_matched_cos"].astype(np.float64).mean(),
        "unmatched": j["pools"]["all"]["unmatched_cos"],
        "top1": 100 * (z["all_rank"] < 1).mean(),
        "top1_full": 100 * (z["fully_measured_rank"] < 1).mean(),
        "n": len(z["all_rank"]), "n_full": len(z["fully_measured_rank"])})
r = {c: pd.DataFrame(v) for c, v in runs.items()}

check("held-out profiles", r["overlap"]["n"].iloc[0], 16912, 0)
check("chance top-1 (%)", 100 / r["overlap"]["n"].iloc[0], 0.0059, 4)
check("overlap: matched cosine, mean", r["overlap"]["matched"].mean(), 0.99549, 5)
check("overlap: matched cosine, SD", r["overlap"]["matched"].std(), 0.00001, 5)
check("overlap: unmatched cosine, mean", r["overlap"]["unmatched"].mean(), 0.30952, 5)
check("overlap: top-1 (%)", r["overlap"]["top1"].mean(), 98.06, 2)
check("  lowest draw", r["overlap"]["top1"].min(), 98.05, 2)
check("  highest draw", r["overlap"]["top1"].max(), 98.07, 2)
check("disjoint: matched cosine, mean", r["disjoint"]["matched"].mean(), 0.99139, 5)
check("disjoint: matched cosine, SD", r["disjoint"]["matched"].std(), 0.00002, 5)
check("disjoint: top-1 (%)", r["disjoint"]["top1"].mean(), 89.38, 2)
check("  lowest draw", r["disjoint"]["top1"].min(), 89.29, 2)
check("  highest draw", r["disjoint"]["top1"].max(), 89.45, 2)
check("pattern-only: top-1 (%)", r["pattern_only"]["top1"].iloc[0], 0.438, 3)
check("profiles >= 99% measured", r["pattern_only"]["n_full"].iloc[0], 7845, 0)
check("  pattern-only top-1 (%)", r["pattern_only"]["top1_full"].iloc[0], 0.025, 3)
check("  chance top-1 (%)", 100 / r["pattern_only"]["n_full"].iloc[0], 0.0127, 4)
report(start)

                       quantity   recomputed  manuscript match
              held-out profiles 16912.000000 16912.00000   yes
               chance top-1 (%)     0.005913     0.00590   yes
  overlap: matched cosine, mean     0.995494     0.99549   yes
    overlap: matched cosine, SD     0.000008     0.00001   yes
overlap: unmatched cosine, mean     0.309523     0.30952   yes
             overlap: top-1 (%)    98.058600    98.06000   yes
                    lowest draw    98.048700    98.05000   yes
                   highest draw    98.066500    98.07000   yes
 disjoint: matched cosine, mean     0.991395     0.99139   yes
   disjoint: matched cosine, SD     0.000021     0.00002   yes
            disjoint: top-1 (%)    89.376400    89.38000   yes
                    lowest draw    89.285700    89.29000   yes
                   highest draw    89.445400    89.45000   yes
        pattern-only: top-1 (%)     0.437560     0.43800   yes
       profiles >= 99% measured  7845.000000  7845.0000

### Reconstruction of withheld CpGs (Fig. 2c,d)

Per-profile MSE over the measured CpGs withheld from the encoder input, for the model and three
controls under identical masks, on 5,000 held-out profiles. The correlation is computed on the
stored random sample of withheld positions shown in Fig. 2c. Bootstrap intervals are for the paired
per-profile control-minus-model difference (10,000 resamples, seed 0).

In [6]:
start = len(checks)
R = PRE / "reconstruction_withheld"
rs = json.loads((R / "reconstruction_withheld_summary.json").read_text())
pm = pd.read_csv(R / "per_sample_metrics.csv")
sc = np.load(R / "scatter_sample.npz")

check("profiles", len(pm), 5000, 0)
check("withheld positions (x 1e8)", rs["n_withheld_positions_total"] / 1e8, 1.12, 2)
check("Pearson r, plotted sample", np.corrcoef(sc["observed"].astype(np.float64), sc["predicted"].astype(np.float64))[0, 1], 0.972, 3)
check("plotted positions (x 1e5)", len(sc["observed"]) / 1e5, 5.0, 1)
for col, name, paper in [("model_mse", "model", 0.0056), ("b_mean_mse", "per-CpG mean", 0.0306),
                         ("b_random_mse", "random vectors", 0.0505), ("b_shuffle_mse", "shuffled CLS", 0.0552)]:
    check(f"mean MSE, {name}", pm[col].mean(), paper, 4)

rng = np.random.default_rng(0)
idx = rng.integers(0, len(pm), size=(10000, len(pm)))
for col in ("b_mean_mse", "b_random_mse", "b_shuffle_mse"):
    d = (pm[col] - pm["model_mse"]).to_numpy()
    lo, hi = np.percentile(d[idx].mean(axis=1), [2.5, 97.5])
    print(f"{col:14s} - model: {d.mean():.4f}  95% CI {lo:.4f} to {hi:.4f}  above zero: {lo > 0}")
    check(f"CI of {col} - model above zero", lo > 0, True, 0)
report(start)

b_mean_mse     - model: 0.0250  95% CI 0.0245 to 0.0255  above zero: True


b_random_mse   - model: 0.0449  95% CI 0.0443 to 0.0454  above zero: True


b_shuffle_mse  - model: 0.0496  95% CI 0.0486 to 0.0505  above zero: True
                              quantity  recomputed manuscript match
                              profiles 5000.000000       5000   yes
            withheld positions (x 1e8)    1.124000       1.12   yes
             Pearson r, plotted sample    0.972100      0.972   yes
             plotted positions (x 1e5)    4.999000        5.0   yes
                       mean MSE, model    0.005629     0.0056   yes
                mean MSE, per-CpG mean    0.030611     0.0306   yes
              mean MSE, random vectors    0.050509     0.0505   yes
                mean MSE, shuffled CLS    0.055190     0.0552   yes
   CI of b_mean_mse - model above zero    1.000000       True   yes
 CI of b_random_mse - model above zero    1.000000       True   yes
CI of b_shuffle_mse - model above zero    1.000000       True   yes


## 3. Frozen representations (Fig. 3, Supplementary Table S2)

Probes use the original partition of the downstream collection: fitted on the training and
validation profiles, evaluated on the test profiles, features standardised on the fitted profiles,
ridge $\alpha = 1$. The classification probes keep classes with at least 30 profiles.

In [7]:
start = len(checks)
meta = pd.read_csv(PRE / "metadata.csv")
cls = np.load(PRE / "embeddings_cls.npy").astype(np.float64)
mean_pooled = np.load(PRE / "embeddings_mean.npy").astype(np.float64)
age, split = meta["age"].to_numpy(float), meta["split"].to_numpy()
fit, ev = split != "test", split == "test"
check("profiles", len(meta), 10988, 0)

def ridge(X, tr, te):
    s = StandardScaler().fit(X[tr])
    p = Ridge(alpha=1.0).fit(s.transform(X[tr]), age[tr]).predict(s.transform(X[te]))
    return r2_score(age[te], p), np.median(np.abs(p - age[te]))

r2, medae = ridge(cls, fit, ev)
check("age probe, CLS: R2", r2, 0.866, 3)
check("age probe, CLS: MedAE", medae, 5.18, 2)
r2, medae = ridge(mean_pooled, fit, ev)
check("age probe, mean-pooled: R2", r2, 0.795, 3)
check("age probe, mean-pooled: MedAE", medae, 6.53, 2)
check("probe fitting profiles", fit.sum(), 8724, 0)
check("probe evaluation profiles", ev.sum(), 2264, 0)

def classify(column):
    labels = meta[column].astype(str).to_numpy()
    counts = pd.Series(labels).value_counts()
    keep = np.isin(labels, counts[counts >= 30].index) & (labels != "nan")
    tr, te = keep & fit, keep & ev
    enc = LabelEncoder().fit(labels[keep])
    s = StandardScaler().fit(cls[tr])
    model = LogisticRegression(max_iter=2000).fit(s.transform(cls[tr]), enc.transform(labels[tr]))
    acc = balanced_accuracy_score(enc.transform(labels[te]), model.predict(s.transform(cls[te])))
    return acc, len(enc.classes_), te.sum()

for column, paper in [("tissue_type", (0.936, 37, 2197, 0.027)), ("dataset", (0.956, 85, 2006, 0.012))]:
    acc, k, n = classify(column)
    check(f"{column} probe: balanced accuracy", acc, paper[0], 3)
    check(f"  classes", k, paper[1], 0)
    check(f"  evaluation profiles", n, paper[2], 0)
    check(f"  chance", 1 / k, paper[3], 3)

unit = cls / (np.linalg.norm(cls, axis=1, keepdims=True) + 1e-9)
nn = NearestNeighbors(n_neighbors=11).fit(unit).kneighbors(unit)[1][:, 1:]
tissue, dataset = meta["tissue_type"].astype(str).to_numpy(), meta["dataset"].astype(str).to_numpy()
same_t, same_d = tissue[nn] == tissue[:, None], dataset[nn] == dataset[:, None]
check("10-NN sharing tissue and dataset (%)", 100 * (same_t & same_d).mean(), 84.6, 1)
check("10-NN sharing tissue only (%)", 100 * (same_t & ~same_d).mean(), 7.5, 1)
check("10-NN sharing dataset only (%)", 100 * (same_d & ~same_t).mean(), 1.5, 1)

def geometry(X):
    s = np.linalg.svd(X - X.mean(0), compute_uv=False)
    p = s**2 / (s**2).sum()
    unit = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    i = np.random.default_rng(0).choice(len(X), 4000, replace=False)
    g = unit[i] @ unit[i].T
    return np.exp(-(p * np.log(p + 1e-12)).sum()), g[np.triu_indices(4000, 1)].mean(), np.linalg.norm(X, axis=1).mean()

for name, X, paper in [("CLS", cls, (18.12, 0.500, 10.01)), ("mean-pooled", mean_pooled, (3.40, 0.867, 5.36))]:
    rank, aniso, norm = geometry(X)
    check(f"{name}: effective rank", rank, paper[0], 2)
    check(f"{name}: mean cosine anisotropy", aniso, paper[1], 3)
    check(f"{name}: mean L2 norm", norm, paper[2], 2)
report(start)

                            quantity  recomputed  manuscript match
                            profiles 10988.00000   10988.000   yes
                  age probe, CLS: R2     0.86632       0.866   yes
               age probe, CLS: MedAE     5.17560       5.180   yes
          age probe, mean-pooled: R2     0.79527       0.795   yes
       age probe, mean-pooled: MedAE     6.53370       6.530   yes
              probe fitting profiles  8724.00000    8724.000   yes
           probe evaluation profiles  2264.00000    2264.000   yes
tissue_type probe: balanced accuracy     0.93579       0.936   yes
                             classes    37.00000      37.000   yes
                 evaluation profiles  2197.00000    2197.000   yes
                              chance     0.02703       0.027   yes
    dataset probe: balanced accuracy     0.95582       0.956   yes
                             classes    85.00000      85.000   yes
                 evaluation profiles  2006.00000    2006.000  

### Study-held-out age probe (Fig. 3e)

For every study with at least 100 profiles, the ridge probe is fitted on all other profiles and
evaluated on that study. The summary uses the studies with an age SD of at least 10 years. The
in-distribution reference is a random 80/20 split of all profiles (seed 0).

In [8]:
start = len(checks)
rng = np.random.default_rng(0)
perm = rng.permutation(len(cls)); cut = int(0.8 * len(cls))
tr, te = np.zeros(len(cls), bool), np.zeros(len(cls), bool)
tr[perm[:cut]], te[perm[cut:]] = True, True
r2, medae = ridge(cls, tr, te)
check("in-distribution 80/20: R2", r2, 0.856, 3)
check("in-distribution 80/20: MedAE", medae, 5.05, 2)
check("in-distribution 80/20: evaluation profiles", te.sum(), 2198, 0)

rows = []
counts = pd.Series(dataset).value_counts()
for study in counts[counts >= 100].index:
    held = dataset == study
    r2, medae = ridge(cls, ~held, held)
    rows.append({"study": study, "n": held.sum(), "age_sd": age[held].std(), "R2": r2, "MedAE": medae})
lodo = pd.DataFrame(rows)
wide = lodo[lodo["age_sd"] >= 10].sort_values("R2", ascending=False)
print(wide.round(3).to_string(index=False), "\n")

rho_sd, p_sd = spearmanr(wide["R2"], wide["age_sd"])
rho_n, p_n = spearmanr(wide["R2"], wide["n"])
check("eligible studies", len(wide), 11, 0)
check("median study-held-out R2", wide["R2"].median(), 0.261, 3)
check("median study-held-out MedAE", wide["MedAE"].median(), 8.32, 2)
check("studies with negative R2", (wide["R2"] < 0).sum(), 3, 0)
check("Spearman rho, R2 vs age SD", rho_sd, 0.28, 2)
check("  P", p_sd, 0.40, 2)
check("Spearman rho, R2 vs study size", rho_n, 0.26, 2)
check("  P", p_n, 0.43, 2)
report(start)

       study   n  age_sd     R2  MedAE
    GSE15745 151  22.508  0.791  5.771
    GSE42861 335  11.461  0.680  4.253
    GSE41037 394  17.712  0.622  7.835
E-GEOD-36194 433  27.522  0.607 10.828
    GSE34035 118  10.489  0.422  5.050
   TCGA_KIRC 356  12.114  0.261  6.856
E-GEOD-40279 393  15.081  0.203  9.652
E-GEOD-64511 222  25.964  0.068 16.843
   TCGA_BRCA 113  14.628 -0.257  8.319
E-GEOD-53740 193  10.275 -0.529 10.462
E-GEOD-56105 282  13.887 -4.630 17.873 

                                  quantity  recomputed  manuscript match
                 in-distribution 80/20: R2     0.85625       0.856   yes
              in-distribution 80/20: MedAE     5.04900       5.050   yes
in-distribution 80/20: evaluation profiles  2198.00000    2198.000   yes
                          eligible studies    11.00000      11.000   yes
                  median study-held-out R2     0.26071       0.261   yes
               median study-held-out MedAE     8.31870       8.320   yes
                  s

## Summary

In [9]:
table = pd.DataFrame(checks)
print(f"{(table['match'] == 'yes').sum()} of {len(table)} manuscript values reproduced")
mismatch = table[table["match"] != "yes"]
if len(mismatch):
    print(mismatch.to_string(index=False))

143 of 143 manuscript values reproduced
